In [1]:
from google import genai
import os
from dotenv import load_dotenv
import json
import chromadb

# Load environment variables
load_dotenv()

# Initialize Gemini client
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# Initialize Chroma
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="kb_gemini")


# Embedding function
def embed_text(text: str):
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )
    return response.embeddings[0].values


# Load knowledge base and index into Chroma
with open("knowledge_base.json", "r") as f:
    kb = json.load(f)

for i, item in enumerate(kb):
    embedding = embed_text(item["text"])

    collection.add(
        ids=[str(i)],
        embeddings=[embedding],
        documents=[item["text"]],
        metadatas=[{"source": item["source"]}]
    )


# Retrieval function
def retrieve(question, k=3):
    q_emb = embed_text(question)

    results = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=["documents", "metadatas"]
    )

    passages = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        passages.append(f"[Source: {meta['source']}]\n{doc}")

    return passages


# Prompt builder
def build_prompt(question, passages):
    context = "\n\n".join(passages)

    return f"""
You are a retrieval-based QA system.

Rules:
- Answer ONLY using the provided context.
- If the answer is not in the context, say: "I don't know."
- Always cite sources like [Source: name].

Context:
{context}

Question:
{question}

Answer:
"""


# Generation function
def generate(prompt):
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )
    return response.text


# Full RAG pipeline
def ask(question, k=3):
    passages = retrieve(question, k)
    prompt = build_prompt(question, passages)
    answer = generate(prompt)

    return {
        "question": question,
        "retrieved_passages": passages,
        "answer": answer
    }

In [4]:
print(ask("How long do I have to get a full refund?"))

{'question': 'How long do I have to get a full refund?', 'retrieved_passages': ['[Source: policy.md]\nOur refund policy allows a full refund within 30 days of purchase, provided the item is unused and in its original packaging. After 30 days, only store credit is offered.', '[Source: policy.md]\nTo cancel your subscription, open Account Settings and choose End Plan. Cancellation takes effect at the end of the current billing period; no partial refunds are given for the remaining days.', '[Source: policy.md]\nPremium plan members get priority support, with a guaranteed first response within four business hours. Standard members receive a response within two business days.'], 'answer': 'You have within 30 days of purchase to get a full refund, provided the item is unused and in its original packaging [Source: policy.md].'}


In [5]:
print(ask("How do I reset my password?"))

{'question': 'How do I reset my password?', 'retrieved_passages': ["[Source: it.md]\nReset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.", "[Source: handbook.md]\nTo power up a device that won't turn on, hold the power button for ten seconds, then connect the charger and wait two minutes before trying again.", '[Source: policy.md]\nPremium plan members get priority support, with a guaranteed first response within four business hours. Standard members receive a response within two business days.'], 'answer': "Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security. [Source: it.md]"}


In [6]:
print(ask("What is the company's stock price today?"))

{'question': "What is the company's stock price today?", 'retrieved_passages': ['[Source: policy.md]\nPremium plan members get priority support, with a guaranteed first response within four business hours. Standard members receive a response within two business days.', '[Source: handbook.md]\nEmployees may park in lot B after 6pm on weekdays. Lot A is reserved for visitors during business hours and requires a pass from reception.', '[Source: policy.md]\nTo cancel your subscription, open Account Settings and choose End Plan. Cancellation takes effect at the end of the current billing period; no partial refunds are given for the remaining days.'], 'answer': "I don't know."}


In [3]:
print(ask("How long do I have to get a full refund?",k=1))

{'question': 'How long do I have to get a full refund?', 'retrieved_passages': ['[Source: policy.md]\nOur refund policy allows a full refund within 30 days of purchase, provided the item is unused and in its original packaging. After 30 days, only store credit is offered.'], 'answer': 'You have within 30 days of purchase to get a full refund, provided the item is unused and in its original packaging. [Source: policy.md]'}
